# Mini Lab: Logistic Regression & SVM

### Names: Joshua Samuel, Avery John, Zachary Jennings
### DS 7331


In [19]:
# Load libraries
import time

import pandas as pd
from clean import load_raw, clean_mushrooms
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                             classification_report, f1_score, confusion_matrix)

In [20]:
# Load clean dataset
df = clean_mushrooms(load_raw())
X = df.drop(columns="class") # Predictors (every column but class)
y = (df["class"] == "poisonous").astype(int)   # Target variable (1 if poisonous, 0 if edible)

In [21]:
# Number of NA values in each column
df.isna().sum()

cap-diameter            0
cap-shape               0
cap-surface             0
cap-color               0
does-bruise-or-bleed    0
gill-attachment         0
gill-spacing            0
gill-color              0
stem-height             0
stem-width              0
stem-root               0
stem-surface            0
stem-color              0
veil-type               0
veil-color              0
has-ring                0
ring-type               0
spore-print-color       0
habitat                 0
season                  0
class                   0
has-stem                0
size-score              0
dtype: int64

## Question 1
Q: Create a logistic regression model and a support vector machine model for the
classification task involved with your dataset. Assess how well each model performs (use
80/20 training/testing split for your data). Adjust parameters of the models to make them more
accurate. If your dataset size requires the use of stochastic gradient descent, then linear kernel
only is fine to use

#### Baseline Logistic Regression Model

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,test_size=0.20, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(48738, 22) (12185, 22) (48738,) (12185,)


In [23]:
# Split Feature Columns
num_cols = X.select_dtypes("number").columns
cat_cols = X.select_dtypes(exclude="number").columns

In [24]:
# Preprocessing Step
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

In [25]:
# Logistic Regression - Preprocessing / Modeling
log_reg = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])

In [26]:
# Fit and Evaluate

start_time = time.time()
log_reg.fit(X_train, y_train)
baseline_log_training_time = time.time() - start_time

preds = log_reg.predict(X_test)

print(f"Training time: {baseline_log_training_time:.2f} seconds")
print("Accuracy:", accuracy_score(y_test, preds))
print("Poisonous recall:", recall_score(y_test, preds))
print("Confusion matrix:\n", confusion_matrix(y_test, preds))
print(classification_report(y_test, preds, target_names=["edible", "poisonous"]))

Training time: 0.61 seconds
Accuracy: 0.8640131308986458
Poisonous recall: 0.8793895391909913
Confusion matrix:
 [[4593  843]
 [ 814 5935]]
              precision    recall  f1-score   support

      edible       0.85      0.84      0.85      5436
   poisonous       0.88      0.88      0.88      6749

    accuracy                           0.86     12185
   macro avg       0.86      0.86      0.86     12185
weighted avg       0.86      0.86      0.86     12185



**Actually Edible, Predicted Edible:** 4,593 (Correct)

**Actually Edible, Predicted Poisonous:** 843 (False Positive)

**Actually Poisonous, Predicted Edible:** 814 (False Negative)

**Actually Poisonous, Predicted Poisonous:** 5,935 (Correct)

#### Baseline SVM Model

In [27]:
# SVM - Preprocessing / Modeling
svm = Pipeline([("prep", preprocess), ("model", SVC(kernel="rbf", random_state=42))])

In [28]:
# Fit and Evaluate

start_time = time.time()
svm.fit(X_train, y_train)
baseline_svm_training_time = time.time() - start_time

svm_preds = svm.predict(X_test)

print(f"Training time: {baseline_svm_training_time:.2f} seconds")
print("Accuracy:", accuracy_score(y_test, svm_preds))
print("Poisonous recall:", recall_score(y_test, svm_preds))
print("Confusion matrix:\n", confusion_matrix(y_test, svm_preds))
print(classification_report(y_test, svm_preds, target_names=["edible", "poisonous"]))

Training time: 12.83 seconds
Accuracy: 0.9996717275338531
Poisonous recall: 1.0
Confusion matrix:
 [[5432    4]
 [   0 6749]]
              precision    recall  f1-score   support

      edible       1.00      1.00      1.00      5436
   poisonous       1.00      1.00      1.00      6749

    accuracy                           1.00     12185
   macro avg       1.00      1.00      1.00     12185
weighted avg       1.00      1.00      1.00     12185



**Actually Edible, Predicted Edible:** 5,432 (Correct)

**Actually Edible, Predicted Poisonous:** 4 (False Positive)

**Actually Poisonous, Predicted Edible:** 0 (False Negative)

**Actually Poisonous, Predicted Poisonous:** 6,749 (Correct)

#### Tuning - Logistic Regression

In [29]:
# Tuning - Logistic Regression
lr_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"],
}
lr_search = GridSearchCV(log_reg, lr_grid, cv=5, scoring="recall", n_jobs=-1)

start_time = time.time()
lr_search.fit(X_train, y_train)
grid_logistic_time = time.time() - start_time

print(f"Grid search time: {grid_logistic_time:.2f} seconds")
print("Best parameters:", lr_search.best_params_)
print("Best CV recall:", lr_search.best_score_)

Grid search time: 8.22 seconds
Best parameters: {'model__C': 100, 'model__class_weight': None}
Best CV recall: 0.8735229192128054


#### Tuning - SVM

In [30]:
# Tuning - SVM
sub = X_train.sample(10000, random_state=42) # Using a training subsample to conserve time, then refit later on full set
y_sub = y_train.loc[sub.index] 

svm_grid = {
    "model__C": [0.1, 1, 10],
    "model__gamma": ["scale", "auto"],
    "model__class_weight": [None, "balanced"]
}
svm_search = GridSearchCV(svm, svm_grid, cv=3, scoring="recall", n_jobs=-1)

start_time = time.time()
svm_search.fit(sub, y_sub)
svm_time = time.time() - start_time

print(f"SVM Grid search time: {svm_time:.2f} seconds")
print("SVM Best params:", svm_search.best_params_)
print("SVM Best CV recall:", svm_search.best_score_)

# Refit best settings on full training set
best_svm = svm_search.best_estimator_

start_time = time.time()
best_svm.fit(X_train, y_train) # Refit on the full training set, not just the 10k subsample
final_training_time = time.time() - start_time
print(f"SVM Final training time: {final_training_time:.2f} seconds")

SVM Grid search time: 14.32 seconds
SVM Best params: {'model__C': 10, 'model__class_weight': None, 'model__gamma': 'scale'}
SVM Best CV recall: 0.999632285346571
SVM Final training time: 5.60 seconds


In [31]:
# Final Test Evaluation
best_lr = lr_search.best_estimator_
best_svm = svm_search.best_estimator_

for name, model in [("LR (default)", log_reg), ("LR (tuned)", best_lr),
                    ("SVM (default)", svm), ("SVM (tuned)", best_svm)]:
    p = model.predict(X_test)
    print(name, f"acc={accuracy_score(y_test, p):.4f}",
          f"recall={recall_score(y_test, p):.4f}",
          f"precision={precision_score(y_test, p):.4f}")
    print(confusion_matrix(y_test, p))

LR (default) acc=0.8640 recall=0.8794 precision=0.8756
[[4593  843]
 [ 814 5935]]
LR (tuned) acc=0.8647 recall=0.8806 precision=0.8758
[[4593  843]
 [ 806 5943]]
SVM (default) acc=0.9997 recall=1.0000 precision=0.9994
[[5432    4]
 [   0 6749]]
SVM (tuned) acc=1.0000 recall=1.0000 precision=1.0000
[[5436    0]
 [   0 6749]]


In [32]:
# Save each model's predictions once, then compare them
models = [
    ("Baseline Logistic Regression", log_reg, baseline_log_training_time),
    ("Baseline SVM", svm, baseline_svm_training_time),
    ("Tuned Logistic Regression", best_lr, grid_logistic_time),
    ("Tuned SVM", best_svm, svm_time + final_training_time)
]

rows = []
for name, model, seconds in models:
    predictions = model.predict(X_test)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Poisonous Precision": precision_score(y_test, predictions),
        "Poisonous Recall": recall_score(y_test, predictions),
        "Poisonous F1": f1_score(y_test, predictions),
        "Total Training / Tuning Time (sec)": seconds
    })

comparison_table = pd.DataFrame(rows)
comparison_table.round(3)

,Model,Accuracy,Poisonous Precision,Poisonous Recall,Poisonous F1,Total Training / Tuning Time (sec)
0,Baseline Logistic Regression,0.864,0.876,0.879,0.878,0.607
1,Baseline SVM,1.000,0.999,1.000,1.000,12.825
2,Tuned Logistic Regression,0.865,0.876,0.881,0.878,8.222
3,Tuned SVM,1.000,1.000,1.000,1.000,19.916


- **Accuracy:** Percentage of all mushrooms classified correctly.
- **Poisonous precision:** Of the mushrooms predicted poisonous, the percentage that actually were poisonous.
- **Poisonous recall:** Of the mushrooms that actually were poisonous, the percentage identified as poisonous. A missed poisonous mushroom would be predicted edible.
- **Poisonous F1:** One score that combines poisonous precision and recall.
- **Total training / tuning time:** For baseline models, time to fit one model. For tuned models, grid search time plus the final full-training fit. The SVM grid search used a 10,000-row training subset, while the logistic regression grid search used all training rows, so these times reflect different amounts of work.

### Question 1 Write-up

Both models were evaluated first on the default hyperparameters to establish a baseline for comparison purposes once we move into tuning each model. The logistic regression model acheived an 86.4% accuracy rate, as well as an 87.9% recall on poisonous mushrooms (sensitivity) and 87.6% precision. Of 6,749 poisonous mushrooms in the test set, 814 (12.1%) were misclassified as edible, and 843 edible mushrooms were misclassified as poisonous.

The support vector machine (SVM) model achieved an accuracy rate of 99.97%, along with 100% perfect recall on poisonous mushrooms and 99.9% precision. Only 4 misclassifications occurred across the entire test set, and all of them were edible mushrooms being misclassified as poisonous. There were zero poisonous mushrooms that were missed.

Both models were then tuned using the GridSearchCV function with cross-validation on the training set, holding out the test set until final evaluation. For the logistic regression model, C and class_weight were searched via 5-fold cross-validation, using recall (sensitivity) as the scoring metric in order to align tuning directly with our goal to minimize missed poisonous mushrooms. The best combination (C=100, class_weight=None) achieved a cross-validated recall of 87.4%, and on the held-out test set produced a modest improvement over the default configuration. Accuracy slightly increased (86.40% to 86.47%), recall improved (87.94% to 88.06%), and the number of missed poisonous mushrooms dropped from 814 to 806. The gain is marginally small, reinforcing that the logistic regression model's linear decision boundary is still the main constraint on its performance for this dataset, even after tuning. 

For the SVM model, C and class_weight were searched via a 3-fold cross validation on a 10,000 row subsample of the training set. With a training set of 48,738 rows, the RBF-kernel SVM remained computationally feasible to train directly, **so stochastic gradient descent was not required**. Subsampling was done to reduce training time, as running this code with the entire training set would cost time. Later, it was refit on the full training set, and the best combination (C=10, class_weight = "None") achieved a 99.9% cross-validated recall. This confirms that the default hyperparamters were already near-optimal for this dataset. The SVM's strength comes from its non-linear kernel rather than from tuning parameters, as the non-linear kernel is better suited for mapping out the decision boundary of the subpopulations.

Given our overall objective is to minimize false negatives, as a missed poisonous mushroom carries severe health risk, the SVM's perfect recall score makes it the stronger candidate before and after tuning is applied. 

## Question 2
Q: Discuss the advantages of each model for each classification task. Does one type
of model offer superior performance over another in terms of prediction accuracy? In terms of
training time or efficiency? Explain in detail.

Logistic regression was fast and gave us coefficients we could interpret. Its baseline test accuracy was 86.4%, increasing to 86.47% after tuning. The RBF SVM could learn a more flexible boundary between edible and poisonous mushrooms. It made four mistakes before tuning and none after tuning on the 12,185 test mushrooms. In both cases, it identified every poisonous mushroom in this test set.

Logistic regression took on average less than 1 second for a single baseline fit, compared with roughly 15 seconds for the baseline SVM. The tuned models took longer because grid search fit multiple models. Overall, the tuned SVM performed best on this test set, while logistic regression was faster to train and easier to explain.

The SVM's non-linear kernel provides a decisive accuracy advantage that outweighs its higher training cost, given the project's priority on minimizing missed poisonous mushrooms. Logistic regression remains the more efficient and interpretable choice, and it would be the more practical option if we were handling bigger data or if we prioritized model transparency more than maximizing model accuracy.

## Question 3
Q: Use the weights from logistic regression to interpret the importance of different
features for each classification task. Explain your interpretation in detail. Why do you think
some variables are more important?

In [33]:
# Extract coefficients
feature_names = best_lr["prep"].get_feature_names_out()
coefs = pd.Series(best_lr["model"].coef_[0], index=feature_names).sort_values()

pd.set_option("display.max_rows", None)

print("Top 15 features pushing toward POISONOUS (largest positive coefficients):")
print(coefs.tail(15).sort_values(ascending=False))

print("\nTop 15 features pushing toward EDIBLE (largest negative coefficients):")
print(coefs.head(15))

print(coefs[coefs.index.str.contains("has-stem|size-score")])

Top 15 features pushing toward POISONOUS (largest positive coefficients):
cat__ring-type_zone              16.272111
cat__spore-print-color_black     12.468577
cat__stem-root_club              12.400208
cat__veil-color_purple           11.699261
cat__veil-type_universal         11.012229
cat__stem-surface_grooves        10.944865
cat__stem-surface_shiny           8.158933
cat__stem-root_rooted             8.110227
cat__habitat_paths                7.290447
cat__veil-color_red               7.232995
cat__veil-color_brown             7.018002
cat__ring-type_evanescent         5.576765
cat__veil-color_Missing           4.871455
cat__spore-print-color_purple     3.758698
cat__stem-color_black             3.627051
dtype: float64

Top 15 features pushing toward EDIBLE (largest negative coefficients):
cat__veil-color_yellow         -17.063947
cat__ring-type_movable         -12.252223
cat__veil-color_white          -10.557740
cat__stem-color_blue            -7.974092
cat__veil-type_Missing    

In [34]:
# SANITY CHECK -- Checking value counts for specific categories
for col in ["ring-type", "spore-print-color", "stem-root", "veil-color", "veil-type", "stem-surface", "habitat", "stem-color", "cap-surface"]:
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print()

--- ring-type ---
ring-type
none          48215
Missing        2471
evanescent     2435
zone           2118
large          1427
flaring        1399
pendant        1265
grooved        1240
movable         353
Name: count, dtype: int64

--- spore-print-color ---
spore-print-color
Missing    54597
black       2118
pink        1259
white       1212
brown       1031
gray         353
purple       182
green        171
Name: count, dtype: int64

--- stem-root ---
stem-root
Missing    51536
swollen     3177
bulbous     3177
rooted      1412
fibrous      915
club         706
Name: count, dtype: int64

--- veil-color ---
veil-color
Missing    53510
white       5474
yellow       527
brown        525
purple       353
black        353
red          181
Name: count, dtype: int64

--- veil-type ---
veil-type
Missing      57746
universal     3177
Name: count, dtype: int64

--- stem-surface ---
stem-surface
Missing    38122
smooth      6025
scaly       4940
fibrous     4396
sticky      2644
grooves     1

### Question 3 Write-Up

We used the tuned logistic regression model to see which mushroom characteristics had the strongest weights. Our target is **1 = poisonous**, so a positive coefficient pushes the prediction toward poisonous; a negative coefficient pushes it toward edible.

Some of the largest weights were yellow veil color (−17.06), zone ring type (+16.27), club-shaped stem root (+12.40), and black spore print (+12.47). For example, the model associated a zone ring with poisonous mushrooms and a yellow veil with edible mushrooms, after accounting for the other features.

`has-stem` and `size-score`, the two engineered features built from patterns found in the EDA provided some interesting insights. Lacking a stem (has-stem_no, +2.14) pushed strongly toward poisonous, consistent with the finding that stemless "others"-shaped mushrooms were poisonous 100% of the time. Larger overall mushroom size (size-score, -0.18) pushed toward edible, matching the group's finding that edible mushrooms were consistently larger across all three size dimensions - the same three dimensions used to engineer the size-score feature.

These variables are likely more important because they reflect traits tied to mushroom species identity. Since toxicity is itself a species-level trait, physical features that correlate with species lineage (stem development, overall size, veil/root structure) act as natural signals for edibility, even with species labels *not* directly available in the dataset.

## Question 4
Look at the chosen support vectors for the classification task. Do these provide
any insight into the data? Explain.

In [35]:
# Extract the fitted SVM from the tuned SVM pipeline
svm_model = best_svm.named_steps["model"]

# Number of support vectors
print("Total support vectors:", len(svm_model.support_))
print("Support vectors by class:", svm_model.n_support_)
print("Class order:", svm_model.classes_)

# Percentage of training observations used as support vectors
support_vector_pct = len(svm_model.support_) / len(X_train) * 100
print(f"Percent of training data used as support vectors: {support_vector_pct:.2f}%")

# View the original, unencoded rows that became support vectors
support_vector_rows = X_train.iloc[svm_model.support_].copy()
support_vector_rows["Actual Class"] = y_train.iloc[svm_model.support_].map(
    {0: "edible", 1: "poisonous"}
)

support_vector_rows.sample(10, random_state=42)

Total support vectors: 1057
Support vectors by class: [504 553]
Class order: [0 1]
Percent of training data used as support vectors: 2.17%


,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,stem-height,stem-width,...,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season,has-stem,size-score,Actual Class
24950,3.13,convex,smooth,black,no,adnate,close,white,3.46,9.84,...,Missing,Missing,none,none,Missing,woods,summer,yes,7.574,edible
44586,6.63,bell,scaly,brown,no,Missing,Missing,black,11.46,16.28,...,Missing,Missing,none,none,Missing,woods,autumn,yes,19.718,poisonous
53729,9.90,flat,smooth,brown,no,decurrent,close,yellow,5.52,27.52,...,Missing,Missing,none,none,Missing,woods,autumn,yes,18.172,poisonous
32804,3.00,convex,Missing,white,no,adnate,distant,white,4.63,2.66,...,Missing,Missing,none,none,Missing,grasses,autumn,yes,7.896,poisonous
3090,10.82,spherical,scaly,brown,no,Missing,Missing,white,22.91,15.52,...,Missing,Missing,ring,movable,Missing,woods,autumn,yes,35.282,edible
7057,4.03,convex,Missing,green,no,adnate,Missing,green,5.26,5.81,...,Missing,Missing,none,none,Missing,woods,autumn,yes,9.871,edible
14248,5.62,convex,sticky,brown,no,adnate,distant,brown,5.76,5.38,...,Missing,Missing,ring,large,Missing,woods,summer,yes,11.918,edible
40669,2.99,convex,sticky,brown,no,sinuate,close,brown,4.61,5.81,...,Missing,brown,ring,zone,Missing,woods,summer,yes,8.181,poisonous
15306,9.67,convex,scaly,red,no,Missing,Missing,white,8.19,23.14,...,Missing,white,ring,evanescent,Missing,woods,autumn,yes,20.174,edible
44576,7.25,spherical,scaly,brown,no,Missing,Missing,pink,12.12,15.61,...,Missing,Missing,none,none,Missing,woods,autumn,yes,20.931,poisonous


### Question 4 Write-up

The tuned SVM used **1,057 of the 48,738 training mushrooms** as support vectors, or **2.17%**. Of these, **504 were edible** and **553 were poisonous**.

Support vectors are training examples that help set the boundary between the classes. Since only a small portion of the training data were support vectors, many other examples did not directly set the boundary. Support vectors may include mushrooms near the boundary or on the wrong side of its margin; they are not necessarily mistakes.

After looking at a sample of ten rows of our support vectors, we did not notice that many shared characteristics between those that shared the same class. We would need to examine the support vectors more systematically before saying which combinations of features are hardest to separate. The tuned SVM made no errors on this test set, but that does not guarantee it will classify every new mushroom correctly.